In [8]:
import numpy as np
import pandas as pd

def calc_lost_deviance(norm_df: pd.DataFrame, pca_df: pd.DataFrame, n_clusters: int, clusters):

    # calculate total deviance from original dataset
    tot_dev = (norm_df - norm_df.mean()).pow(2).sum().sum()
    
    print("total deviance:", tot_dev)
    
    # calculate deviance from pca
    pca_dev = (pca_df - pca_df.mean()).pow(2).sum().sum()
    
    print("pca deviance:", pca_dev, ";", (pca_dev / tot_dev)*100, "%")
    
    # calculate intra and inter cluster deviation
    intra_cluster_dev = np.zeros(n_clusters)
    inter_cluster_dev = np.zeros(n_clusters)
    
    for cluster in range(n_clusters):
        centroid = pca_df.loc[clusters == cluster].mean()
        
        n_elements = pca_df.loc[clusters == cluster].shape[0]
    
        # compute the deviation as the squared distance from the centroid
        local_intra = (centroid - pca_df.loc[clusters == cluster]).pow(2).sum().sum()
    
        # compute the deviation as the squared distance from the others centrod weighted by cluster size 
        local_inter = n_elements * (centroid - pca_df.mean()).pow(2).sum()
    
        # add to result list
        intra_cluster_dev[cluster] = local_intra
        inter_cluster_dev[cluster] = local_inter
    
    intra_cluster_dev = intra_cluster_dev.sum()
    inter_cluster_dev = inter_cluster_dev.sum()
    
    print((intra_cluster_dev + inter_cluster_dev) / pca_dev)
    
    pca_and_cluster_dev = inter_cluster_dev / pca_dev
    tot_lost_dev = 1 + (intra_cluster_dev - pca_dev) / tot_dev

    print("total deviance lost:", tot_lost_dev * 100, "%")

# Workload characterization
We want to validate a synthetic workload. We can do it through hypotesis test by comparing low level parameters on the server from the real workload and the syntheic one.

# Low level parameters characterization
We performa PCA and clustering on LLC collected from the real traffic.

In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as shc

from scipy.stats import zscore
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering

fname = "wc_real_vmstat.csv"

# read data
df = pd.read_csv(fname)

n_components = 4

# create pca object
pca = PCA(n_components=n_components)

# normalize data using zscore. Fill with 0 constant columns
norm_df = df.apply(zscore, ddof=1).dropna(axis=1)

# create principal components passing all data as array of arrays
principal_components = pca.fit_transform(norm_df.values)

pca_df = pd.DataFrame(principal_components)

pca_df[0] *= -1

# number of cluster
n_clusters = 11

# perform clustering with ward as linkage method
clustering = AgglomerativeClustering(n_clusters=n_clusters)

clusters = clustering.fit_predict(principal_components)

calc_lost_deviance(norm_df, pca_df, n_clusters, clusters)

total deviance: 9252.0
pca deviance: 7683.740686545571 ; 83.04951023071305 %
0.9999999999999997
total deviance lost: 23.793101808006732 %


# High level parameters characterization
We performa PCA and clustering on HLC collected from the real traffic. Results of the clustering will be used to create the synthetic workload.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as shc

from scipy.stats import zscore
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering

fname = "wc_real.csv"

pca_cols = ["elapsed", "bytes", "sentBytes", "Latency", "IdleTime", "Connect"]

# read data
df = pd.read_csv(fname)

n_components = 4

# create pca object
pca = PCA(n_components=n_components)

# normalize data using zscore. Fill with 0 constant columns
norm_df = df[pca_cols].apply(zscore, ddof=1).dropna(axis=1)

# create principal components passing all data as array of arrays
principal_components = pca.fit_transform(norm_df.values)

pca_df = pd.DataFrame(principal_components)

pca_df[0] *= -1

# number of cluster
n_clusters = 18

# perform clustering with ward as linkage method
clustering = AgglomerativeClustering(n_clusters=n_clusters)

clusters = clustering.fit_predict(principal_components)

calc_lost_deviance(norm_df, pca_df, n_clusters, clusters)

In [16]:
import pandas as pd

cluster_file = "cluster.csv"
traffic_file = "wc_real.csv"

cluster_df = pd.read_csv(cluster_file, delimiter=";")
df = pd.read_csv(traffic_file)

df["cluster"] = cluster_df["cluster30"]

n_clusters = int(df["cluster"].max())
print(n_clusters)
df["reqId"] = df["threadName"].apply(lambda x: str(x).split(" ")[0]) + "-" + df["label"].astype(str)

res = {}
for cluster in range(1, n_clusters + 1):
    v = df[df["cluster"] == cluster]["reqId"].value_counts()
    res[cluster] = v.iloc[:10]

request_ids = []
unique = set()
for cluster in res:
    for (i, request_id) in enumerate(res[cluster].keys()):
        if request_id not in unique:
            request_ids.append((request_id, res[cluster].iloc[i]))
            unique.add(request_id)
            break

request_ids, len(request_ids)

30


([('TG4-2', 153),
  ('TG1-10', 110),
  ('TG1-15', 146),
  ('TG1-19', 88),
  ('TG5-19', 22),
  ('TG1-17', 143),
  ('TG1-5', 177),
  ('TG2-11', 7),
  ('TG2-14', 57),
  ('TG3-13', 28),
  ('TG2-2', 26),
  ('TG2-3', 114),
  ('TG5-3', 22),
  ('TG3-14', 79),
  ('TG3-12', 85),
  ('TG2-10', 20),
  ('TG1-4', 92),
  ('TG3-9', 22),
  ('TG3-7', 104),
  ('TG3-6', 60),
  ('TG2-7', 21),
  ('TG2-16', 3),
  ('TG1-11', 2),
  ('TG2-6', 24),
  ('TG3-17', 121),
  ('TG5-15', 15),
  ('TG3-19', 19),
  ('TG3-18', 13),
  ('TG2-20', 84),
  ('TG2-18', 7)],
 30)

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as shc

from scipy.stats import zscore
from sklearn.decomposition import PCA

traffic_fname = "wc_real.csv"
cluster_file = "cluster.csv"

pca_cols = ["elapsed", "bytes", "sentBytes", "Latency", "IdleTime", "Connect"]

# read data
df = pd.read_csv(traffic_fname)
clusters = pd.read_csv(cluster_file, delimiter=";")["cluster30"]

n_components = 4

n_clusters = int(clusters.max())

# create pca object
pca = PCA(n_components=n_components)

# normalize data using zscore. Fill with 0 constant columns
norm_df = df[pca_cols].apply(zscore, ddof=1).dropna(axis=1)

# create principal components passing all data as array of arrays
principal_components = pca.fit_transform(norm_df.values)

pca_df = pd.DataFrame(principal_components)

pca_df[0] *= -1

calc_lost_deviance(norm_df, pca_df, n_clusters, clusters)

total deviance: 48405.0
pca deviance: 46132.208916634314 ; 95.30463571249729 %
0.9400583427679589
total deviance lost: 11.322462114257858 %
